In [16]:
import sys
from pathlib import Path
current_path = Path.cwd().resolve()
print("Current working directory:", current_path)
# Search the current directory and its parents for the project root
project_root = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "src").is_dir()),
    None)
if project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing the src folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())
import importlib
import src.data_processing as dp
importlib.reload(dp)

Current working directory: /Users/ranacopty/Desktop/APS360/aps360-movie-flop-predictor/notebooks
Project root: /Users/ranacopty/Desktop/APS360/aps360-movie-flop-predictor
src exists: True


<module 'src.data_processing' from '/Users/ranacopty/Desktop/APS360/aps360-movie-flop-predictor/src/data_processing.py'>

In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

processed_df = prepare_tmdb_dataset().copy()

# Preserve alignment with the saved synopsis-embedding array
processed_df["_embedding_row"] = np.arange(len(processed_df))

# Reserve the newest films as the chronological test set
cutoff_date = pd.Timestamp("2014-01-01")

development_df = processed_df[
    processed_df["release_date"] < cutoff_date
].copy()

test_df = processed_df[
    processed_df["release_date"] >= cutoff_date
].copy()

# Random, stratified train-validation split within the earlier films
train_df, validation_df = train_test_split(
    development_df,
    test_size=185,
    stratify=development_df["flop"],
    random_state=42,
)

# Optional: clean row indices while retaining _embedding_row
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Total:", len(processed_df))
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

print("\nFlop proportions:")
print("Train:", train_df["flop"].mean())
print("Validation:", validation_df["flop"].mean())
print("Test:", test_df["flop"].mean())

assert len(processed_df) == 857
assert len(train_df) == 559
assert len(validation_df) == 185
assert len(test_df) == 113

output_path = save_processed_dataset(
    processed_df,
    project_root / "data/processed/movies_processed.csv",
)

print("\nSaved:", output_path)

Total: 857
Train: 559
Validation: 185
Test: 113

Flop proportions:
Train: 0.5813953488372093
Validation: 0.5783783783783784
Test: 0.40707964601769914

Saved: /Users/ranacopty/Desktop/APS360/aps360-movie-flop-predictor/data/processed/movies_processed.csv


In [2]:
from src.data_processing import (
    chronological_split,
    prepare_tmdb_dataset,
    save_processed_dataset)
from src.data_processing import save_processed_dataset

In [4]:
processed_df = prepare_tmdb_dataset()
train_df, validation_df, test_df = chronological_split(
    processed_df)
print(len(processed_df))
print(len(train_df), len(validation_df), len(test_df))
output_path = save_processed_dataset(
    processed_df,
    project_root / "data/processed/movies_processed.csv")
print("Saved:", output_path)

857
559 185 113
Saved: /Users/ranacopty/Desktop/APS360/aps360-movie-flop-predictor/data/processed/movies_processed.csv


## Feature inspection

In [6]:
genre_features = [
    column
    for column in processed_df.columns
    if column.startswith("genre_")]
new_features = [
    "release_season",
    "is_sequel_or_remake",
    "major_studio",
    *genre_features,]
print("Number of genre features:", len(genre_features))
print(new_features)

Number of genre features: 20
['release_season', 'is_sequel_or_remake', 'major_studio', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_foreign', 'genre_history', 'genre_horror', 'genre_music', 'genre_mystery', 'genre_romance', 'genre_science_fiction', 'genre_tv_movie', 'genre_thriller', 'genre_war', 'genre_western']


In [7]:
print(processed_df["release_season"].value_counts(dropna=False))
print()

print(processed_df["is_sequel_or_remake"].value_counts(dropna=False))
print()

print(processed_df["major_studio"].value_counts(dropna=False))

release_season
Summer    276
Spring    203
Winter    191
Fall      187
Name: count, dtype: int64

is_sequel_or_remake
0    795
1     62
Name: count, dtype: int64

major_studio
1    651
0    206
Name: count, dtype: int64


In [8]:
display(
    processed_df[
        [
            "title",
            "release_season",
            "is_sequel_or_remake",
            "major_studio",
            *genre_features,
        ]
    ].head(10)
)

,title,release_season,is_sequel_or_remake,major_studio,genre_action,genre_adventure,genre_animation,genre_comedy,genre_crime,genre_documentary,...,genre_history,genre_horror,genre_music,genre_mystery,genre_romance,genre_science_fiction,genre_tv_movie,genre_thriller,genre_war,genre_western
0,Metropolis,Winter,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,Superman,Winter,0,1,1,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,Superman II,Winter,1,1,1,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,The Cotton Club,Winter,0,0,0,0,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
4,Ishtar,Spring,0,1,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,Rambo III,Spring,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0
6,The Abyss,Summer,0,1,1,1,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
7,Tango & Cash,Winter,0,1,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
8,Total Recall,Summer,0,0,1,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
9,Days of Thunder,Summer,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
display(processed_df.head(3))

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,crew,cast,director,lead_actor,rev_budget_ratio,flop,rev_budget_ratio_capped,director_success_ratio,actor_success_ratio,director_prior_count,actor_prior_count,release_season,is_sequel_or_remake,major_studio,genre_action,genre_adventure,genre_animation,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_foreign,genre_history,genre_horror,genre_music,genre_mystery,genre_romance,genre_science_fiction,genre_tv_movie,genre_thriller,genre_war,genre_western,release_year
0,92620000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 878, ""name"": ""Science Fiction""}]",NaN,19,"[{""id"": 312, ""name"": ""man vs machine""}, {""id"": 1001, ""name"": ""underground world""}, {""id"": 1436, ...",de,Metropolis,"In a futuristic city sharply divided between the working class and the city planners, the son of...",32.351527,"[{""name"": ""Paramount Pictures"", ""id"": 4}, {""name"": ""Universum Film (UFA)"", ""id"": 12372}]","[{""iso_3166_1"": ""DE"", ""name"": ""Germany""}]",1927-01-10,650422,153.0,"[{""iso_639_1"": ""xx"", ""name"": ""No Language""}]",Released,There can be no understanding between the hands and the brain unless the heart acts as mediator.,Metropolis,8.0,657,19,"[{""credit_id"": ""52fe420fc3a36847f8000c55"", ""department"": ""Production"", ""gender"": 2, ""id"": 67, ""j...","[{""cast_id"": 10, ""character"": ""Maria"", ""credit_id"": ""52fe420fc3a36847f8000c87"", ""gender"": 1, ""id...",Fritz Lang,Brigitte Helm,0.007022,1,0.007022,1.0,1.00000,0.0,0.0,Winter,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1927
1,55000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {...",NaN,1924,"[{""id"": 83, ""name"": ""saving the world""}, {""id"": 736, ""name"": ""journalist""}, {""id"": 849, ""name"": ...",en,Superman,"Mild-mannered Clark Kent works as a reporter at the Daily Planet alongside his crush, Lois Lane ...",48.507081,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name"": ""Dovemead Films"", ""id"": 51861}, {""name"": ""Film E...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}, {""iso_3166_1"": ""US"", ""name"": ""United States of ...",1978-12-13,300218018,143.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,You'll Believe a Man Can Fly!,Superman,6.9,1022,1924,"[{""credit_id"": ""553929789251414081002335"", ""department"": ""Sound"", ""gender"": 2, ""id"": 491, ""job"":...","[{""cast_id"": 1, ""character"": ""Superman / Clark Kent"", ""credit_id"": ""52fe4322c3a36847f803cdff"", ""...",Richard Donner,Christopher Reeve,5.458509,0,5.458509,10.0,1.00000,1.0,0.0,Winter,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1978
2,54000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {...",NaN,8536,"[{""id"": 83, ""name"": ""saving the world""}, {""id"": 849, ""name"": ""dc comics""}, {""id"": 9663, ""name"": ...",en,Superman II,"Three escaped criminals from the planet Krypton test the Man of Steel's mettle. Led by Gen. Zod,...",30.515175,"[{""name"": ""Warner Bros."", ""id"": 6194}, {""name"": ""Dovemead Films"", ""id"": 51861}, {""name"": ""Film E...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}, {""iso_3166_1"": ""US"", ""name"": ""United States of ...",1980-12-04,190458706,127.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Man of Steel meets his match!,Superman II,6.5,629,8536,"[{""credit_id"": ""52fe44aec3a36847f80a3f8d"", ""department"": ""Camera"", ""gender"": 2, ""id"": 243, ""job""...","[{""cast_id"": 3, ""character"": ""Lex Luthor"", ""credit_id"": ""52fe44aec3a36847f80a3f3b"", ""gender"": 2,...",Richard Lester,Gene Hackman,3.527013,0,3.527013,10.0,6.38125,1.0,2.0,Winter,1,1,1,1,0,0,0,0,0,0,1,0

In [11]:
print("Shape:", processed_df.shape)
print("\nColumns:")
print(processed_df.columns.tolist())

display(processed_df.info())

Shape: (857, 56)

Columns:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'movie_id', 'crew', 'cast', 'director', 'lead_actor', 'rev_budget_ratio', 'flop', 'rev_budget_ratio_capped', 'director_success_ratio', 'actor_success_ratio', 'director_prior_count', 'actor_prior_count', 'release_season', 'is_sequel_or_remake', 'major_studio', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_foreign', 'genre_history', 'genre_horror', 'genre_music', 'genre_mystery', 'genre_romance', 'genre_science_fiction', 'genre_tv_movie', 'genre_thriller', 'genre_war', 'genre_western', 'release_year']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 857 entries, 0 to 856

None

In [12]:
import numpy as np

calculated_ratio = (
    processed_df["revenue"] / processed_df["budget"]
)

print(
    "Ratio matches:",
    np.allclose(
        processed_df["rev_budget_ratio"],
        calculated_ratio,
        equal_nan=True,
    ),
)

expected_flop = (
    processed_df["rev_budget_ratio"] < 2.5
).astype(int)

print(
    "Flop labels match:",
    (
        processed_df["flop"] == expected_flop
    ).all(),
)

Ratio matches: True
Flop labels match: True


In [13]:
print(
    "Chronologically ordered:",
    processed_df["release_date"].is_monotonic_increasing,
)

Chronologically ordered: True
